# 투자거울: 코스피 시계열 지표 및 복기 계산 검증 노트북

이 노트북은 `summary_service`와 `analytics_service`에서 사용되는 핵심 수식을 사전에 프로토타이핑하고 검증합니다.
- 20영업일 이동평균 비교 기반 추세 판별 (상승, 하락, 횡보)
- 1년 최저~최고 범위 내 위치 (%) 계산
- 일간 변동성(표준편차) 및 매수일 기준 위치 산출

In [ ]:
import numpy as np
import pandas as pd

# 샘플 코스피 시계열 데이터 생성 (245영업일)
dates = pd.date_range(end=pd.Timestamp.today(), periods=245, freq='B')
np.random.seed(42)
returns = np.random.normal(0.0003, 0.009, size=245)
price_path = 2500.0 * np.exp(np.cumsum(returns))
df = pd.DataFrame({'date': dates.strftime('%Y-%m-%d'), 'close': price_path})
df.head()

In [ ]:
# 1. 최근 20일 vs 직전 20일 추세 계산 검증
avg_recent = df['close'].iloc[-20:].mean()
avg_prev = df['close'].iloc[-40:-20].mean()
diff_pct = ((avg_recent - avg_prev) / avg_prev) * 100

trend_label = '상승' if diff_pct >= 1.0 else ('하락' if diff_pct <= -1.0 else '횡보')
print(f"최근 20일 평균: {avg_recent:.2f}, 직전 20일 평균: {avg_prev:.2f}, 차이: {diff_pct:+.2f}%, 판정: {trend_label}")

In [ ]:
# 2. 1년 범위 내 위치(%) 계산
min_val = df['close'].min()
max_val = df['close'].max()
latest = df['close'].iloc[-1]
position = ((latest - min_val) / (max_val - min_val)) * 100
print(f"최저: {min_val:.2f}, 최고: {max_val:.2f}, 최신: {latest:.2f}, 위치: {position:.1f}%")